# Football Win Probability Model
**Any two international / club teams · football-data.org API**

### Methodology
- **3-outcome** multinomial logistic regression (Win / Draw / Loss)
- **Competition importance weighting** — World Cup / major tournament matches count more than friendlies
- **Recency decay** — exponential half-life so recent form outweighs old results
- **Squad depth** from the teams endpoint as an auxiliary feature
- **Platt calibration** via `CalibratedClassifierCV`
- **API result caching** — re-fetches only when cache is stale

> Register at https://www.football-data.org for your free API token.

**Quick start:** fill in `API_TOKEN`, `TEAM_A_NAME`, `TEAM_B_NAME`, and `MATCH_DATE` in the Config cell — everything else is automatic.

In [ ]:
# pip install -r requirements.txt
import json
import os
import time as _time
import requests
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── CONFIG — edit these lines only ────────────────────────────────────────────
API_TOKEN       = "YOUR_API_TOKEN_HERE"   # <- paste your football-data.org token

TEAM_A_NAME     = "France"                # predicted win  = Team A wins
TEAM_B_NAME     = "Senegal"               # predicted loss = Team B wins
MATCH_DATE      = "2026-06-16"            # YYYY-MM-DD of the match you are predicting

# Optional: paste numeric team IDs to skip the name-search API call
# Find IDs at https://www.football-data.org/documentation/quickstart
TEAM_A_ID       = None    # e.g. 773 for France
TEAM_B_ID       = None    # e.g. 1991 for Senegal

DATE_FROM       = "2019-01-01"   # earliest match date to include
FORCE_REFRESH   = False          # True = bypass cache and re-fetch from API
CACHE_TTL_HOURS = 24             # hours before cached match data is considered stale
WINDOW          = 12             # rolling window size (matches) for form features
# ──────────────────────────────────────────────────────────────────────────────

if API_TOKEN == "YOUR_API_TOKEN_HERE":
    raise ValueError(
        "API_TOKEN is not set. "
        "Register at https://www.football-data.org and paste your token above."
    )

BASE_URL = "https://api.football-data.org/v4"
HEADERS  = {"X-Auth-Token": API_TOKEN}

COMP_WEIGHTS = {
    "WC":  3.0,   # FIFA World Cup
    "EC":  2.5,   # UEFA Euros
    "CAN": 2.5,   # AFCON
    "UCL": 2.0,   # Champions League
    "UNL": 1.8,   # Nations League
    "WCQ": 1.5,   # World Cup Qualifying
    "FR":  0.6,   # Friendlies
}
DEFAULT_COMP_WEIGHT = 1.0

# National team colours (extend as needed)
TEAM_COLORS = {
    "France": "#002395", "Senegal": "#00853F",
    "Brazil": "#009C3B", "Argentina": "#74ACDF",
    "Germany": "#000000", "Spain": "#AA151B",
    "England": "#CF081F", "Portugal": "#006600",
    "Netherlands": "#FF6600", "Italy": "#003399",
    "Belgium": "#EF3340", "Uruguay": "#5AAFF6",
    "Croatia": "#FF0000", "Morocco": "#C1272D",
    "Japan": "#BC002D", "USA": "#B31942",
    "Mexico": "#006847", "Colombia": "#FCD116",
    "Ecuador": "#FFD100", "Australia": "#00843D",
    "Switzerland": "#FF0000", "Denmark": "#C60C30",
    "Poland": "#DC143C", "Serbia": "#C6363C",
    "Ghana": "#006B3F", "Cameroon": "#007A5E",
    "Nigeria": "#008751", "South Korea": "#003478",
    "Iran": "#239F40", "Saudi Arabia": "#006C35",
    "Qatar": "#8D1B3D", "Tunisia": "#E70013",
    "Canada": "#FF0000", "Wales": "#CF101A",
    "Turkey": "#E30A17", "Ukraine": "#005BBB",
    "Austria": "#ED2939", "Sweden": "#006AA7",
    "Scotland": "#003DA5", "Algeria": "#006233",
    "Egypt": "#CE1126", "Ivory Coast": "#F77F00",
    "Mali": "#009A00", "DR Congo": "#007FFF",
    "South Africa": "#007A4D", "Costa Rica": "#002B7F",
    "Peru": "#D91023", "Chile": "#D52B1E",
    "Paraguay": "#D52B1E", "Bolivia": "#F4E400",
    "Venezuela": "#CF142B", "Honduras": "#0073CF",
    "Panama": "#DA121A", "El Salvador": "#0F47AF",
    "New Zealand": "#00247D", "Indonesia": "#CE1126",
    "China": "#DE2910", "India": "#FF9933",
    "Iraq": "#CE1126", "Jordan": "#007A3D",
    "Russia": "#D52B1E", "Romania": "#002B7F",
    "Hungary": "#CE2939", "Greece": "#0D5EAF",
    "Iceland": "#003897", "Finland": "#003580",
    "Slovakia": "#005B96", "Slovenia": "#003DA5",
    "Albania": "#E41E20", "Georgia": "#FF0000",
    "Czech Republic": "#D7141A", "Norway": "#EF2B2D",
    "Kenya": "#006600", "Ethiopia": "#078930",
    "Zimbabwe": "#006400", "Uganda": "#FCDC04",
    "Thailand": "#A51931", "Vietnam": "#DA251D",
    "UAE": "#00732F", "Bahrain": "#CE1126",
    "Jamaica": "#000000", "Haiti": "#00209F",
    "Cuba": "#002A8F", "Guatemala": "#4997D0",
}
FALLBACK_COLORS = ["#1f77b4", "#ff7f0e"]

def team_color(name, idx):
    return TEAM_COLORS.get(name, FALLBACK_COLORS[idx % 2])

print(f"Config loaded — {TEAM_A_NAME} vs {TEAM_B_NAME} on {MATCH_DATE}")

In [ ]:
# ── API helpers: retry + caching ──────────────────────────────────────────────
def api_get(url, params=None, retries=4):
    """GET with exponential back-off on transient failures."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            r.raise_for_status()
            return r
        except requests.exceptions.HTTPError as e:
            if r.status_code in (429, 503) and attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  HTTP {r.status_code} — retrying in {wait}s...")
                _time.sleep(wait)
            else:
                raise
        except requests.exceptions.RequestException as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  Network error ({e}) — retrying in {wait}s...")
                _time.sleep(wait)
            else:
                raise

def _cache_path(name):
    safe = name.replace(" ", "_").replace("'", "").replace("/", "_")
    return f".cache_{safe}_matches.json"

def _load_cache(name):
    path = _cache_path(name)
    if os.path.exists(path) and not FORCE_REFRESH:
        age_h = ((_time.time() - os.path.getmtime(path)) / 3600)
        if age_h < CACHE_TTL_HOURS:
            with open(path) as f:
                data = json.load(f)
            print(f"  {name}: loaded {len(data)} matches from cache (age {age_h:.1f}h)")
            return data
    return None

def _save_cache(name, data):
    with open(_cache_path(name), "w") as f:
        json.dump(data, f)

# ── team ID lookup ────────────────────────────────────────────────────────────
def search_team_id(name):
    try:
        r     = api_get(f"{BASE_URL}/teams", params={"name": name})
        teams = r.json().get("teams", [])
    except Exception:
        teams = []
    if not teams:
        raise ValueError(
            f"Team '{name}' not found via API search.\n"
            "  → Look up the numeric ID at https://www.football-data.org/documentation/quickstart\n"
            "  → Then set TEAM_A_ID or TEAM_B_ID directly in the Config cell."
        )
    t = teams[0]
    print(f"  Found: {t['name']} (ID {t['id']})")
    return t["id"]

TEAM_IDS = {}
for t_name, override_id in [(TEAM_A_NAME, TEAM_A_ID), (TEAM_B_NAME, TEAM_B_ID)]:
    if override_id is not None:
        TEAM_IDS[t_name] = override_id
        print(f"{t_name}: using provided ID {override_id}")
    else:
        print(f"Searching: {t_name} ...")
        TEAM_IDS[t_name] = search_team_id(t_name)

print("\nTeam IDs:", TEAM_IDS)

In [ ]:
# ── fetch matches (with cache) & squad sizes ───────────────────────────────────
def fetch_matches(team_id, name, date_from=DATE_FROM):
    cached = _load_cache(name)
    if cached is not None:
        return cached
    r = api_get(
        f"{BASE_URL}/teams/{team_id}/matches",
        params={"dateFrom": date_from, "status": "FINISHED", "limit": 100},
    )
    matches = r.json().get("matches", [])
    _save_cache(name, matches)
    print(f"  {name}: fetched {len(matches)} matches from API and cached")
    return matches

def fetch_squad_size(team_id):
    r     = api_get(f"{BASE_URL}/teams/{team_id}")
    squad = r.json().get("squad", [])
    size  = len(squad)
    return size if size > 5 else 23   # guard against empty squad response

raw         = {}
squad_sizes = {}
for name, tid in TEAM_IDS.items():
    print(f"\n{name}:")
    raw[name] = fetch_matches(tid, name)
    try:
        squad_sizes[name] = fetch_squad_size(tid)
        print(f"  squad size = {squad_sizes[name]}")
    except Exception as e:
        squad_sizes[name] = 23
        print(f"  squad fetch failed ({e}), defaulting to 23")

In [ ]:
# ── parse matches: competition weighting + recency decay ──────────────────────
def parse_matches(matches, team_id, comp_weights=COMP_WEIGHTS):
    rows = []
    for m in matches:
        ft = m.get("score", {}).get("fullTime", {})
        if ft.get("home") is None:
            continue

        is_home = m["homeTeam"]["id"] == team_id
        gf = ft["home"] if is_home else ft["away"]
        ga = ft["away"] if is_home else ft["home"]

        comp_code = m.get("competition", {}).get("code", "UNK")
        comp_w    = comp_weights.get(comp_code, DEFAULT_COMP_WEIGHT)

        date      = pd.to_datetime(m["utcDate"][:10])
        days_back = (pd.Timestamp(MATCH_DATE) - date).days
        time_w    = 2 ** (-days_back / 365)

        rows.append({
            "date":    date,
            "comp":    comp_code,
            "is_home": int(is_home),
            "gf":      gf,
            "ga":      ga,
            "gd":      gf - ga,
            "win":     int(gf > ga),
            "draw":    int(gf == ga),
            "loss":    int(gf < ga),
            "points":  3 if gf > ga else (1 if gf == ga else 0),
            "outcome": 2 if gf > ga else (1 if gf == ga else 0),  # 2=win,1=draw,0=loss
            "weight":  comp_w * time_w,
        })

    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    return df

dfs = {n: parse_matches(raw[n], tid) for n, tid in TEAM_IDS.items()}

for name, df in dfs.items():
    w_pct = df["win"].mean()
    d_pct = df["draw"].mean()
    l_pct = df["loss"].mean()
    print(f"{name}: {len(df)} matches | W {w_pct:.0%}  D {d_pct:.0%}  L {l_pct:.0%}")
    print(f"  Competition mix: {dict(df['comp'].value_counts().head(6))}")

In [ ]:
# ── weighted aggregate stats per team ─────────────────────────────────────────
def weighted_stats(df):
    w    = df["weight"].values
    wsum = max(w.sum(), 1e-9)
    return {
        "win_rate":     np.dot(df["win"],    w) / wsum,
        "draw_rate":    np.dot(df["draw"],   w) / wsum,
        "loss_rate":    np.dot(df["loss"],   w) / wsum,
        "gf_per_game":  np.dot(df["gf"],     w) / wsum,
        "ga_per_game":  np.dot(df["ga"],     w) / wsum,
        "gd_per_game":  np.dot(df["gd"],     w) / wsum,
        "pts_per_game": np.dot(df["points"], w) / wsum,
    }

stats = {n: weighted_stats(dfs[n]) for n in TEAM_IDS}

print("Weighted aggregate stats:")
print(pd.DataFrame(stats).T.round(3).to_string())

In [ ]:
# ── build training dataset (3-outcome labels: 0=loss, 1=draw, 2=win) ──────────
FCOLS = ["win_rate", "draw_rate", "gf_rate", "ga_rate", "gd_rate", "pts_rate"]

def windowed_features(df, window=WINDOW):
    out = []
    for i in range(window, len(df)):
        chunk = df.iloc[i - window:i]
        w     = chunk["weight"].values
        ws    = max(w.sum(), 1e-9)
        row   = df.iloc[i]
        out.append({
            "win_rate":       np.dot(chunk["win"],    w) / ws,
            "draw_rate":      np.dot(chunk["draw"],   w) / ws,
            "gf_rate":        np.dot(chunk["gf"],     w) / ws,
            "ga_rate":        np.dot(chunk["ga"],     w) / ws,
            "gd_rate":        np.dot(chunk["gd"],     w) / ws,
            "pts_rate":       np.dot(chunk["points"], w) / ws,
            "date":           row["date"],
            "actual_outcome": row["outcome"],   # 2=win, 1=draw, 0=loss
            "is_home":        row["is_home"],
        })
    return pd.DataFrame(out)

wf = {n: windowed_features(dfs[n]) for n in TEAM_IDS}

def build_diff_dataset(wf_a, wf_b, sq_a, sq_b):
    """Differential form features: Team A stats minus Team B stats."""
    n    = min(len(wf_a), len(wf_b))
    a, b = wf_a.tail(n).reset_index(drop=True), wf_b.tail(n).reset_index(drop=True)
    diff = a[FCOLS].values - b[FCOLS].values
    out  = pd.DataFrame(diff, columns=[f"d_{c}" for c in FCOLS])
    out["home_adv"]   = a["is_home"].values
    out["squad_diff"] = (sq_a - sq_b) / max(sq_a, sq_b, 1)
    out["label"]      = a["actual_outcome"].values
    return out

team_a, team_b = list(TEAM_IDS.keys())
train = build_diff_dataset(
    wf[team_a], wf[team_b],
    squad_sizes[team_a], squad_sizes[team_b]
)
counts = train["label"].value_counts().sort_index().rename({0: "Loss", 1: "Draw", 2: "Win"})
print(f"Training rows : {len(train)}")
print(f"Label balance : {counts.to_dict()}  (for {team_a})")

In [ ]:
# ── 3-class multinomial LR + Platt calibration ────────────────────────────────
FEATS = [c for c in train.columns if c != "label"]
X = train[FEATS].values
y = train["label"].values

class_counts = Counter(y)
min_class_n  = min(class_counts.values())
n_splits     = min(5, min_class_n)

base_clf = LogisticRegression(C=1.5, solver="lbfgs", max_iter=1000, random_state=99)

if n_splits < 2:
    print(f"WARNING: smallest class has only {min_class_n} sample(s). "
          "Fitting without cross-validation.")
    model = Pipeline([("sc", StandardScaler()), ("clf", base_clf)])
    model.fit(X, y)
    print("Model fitted (no CV scores).")
else:
    model = CalibratedClassifierCV(
        estimator=Pipeline([("sc", StandardScaler()), ("clf", base_clf)]),
        cv=n_splits,
        method="sigmoid",
    )
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=99)

    # Multi-class ROC-AUC (one-vs-rest, weighted by class frequency)
    auc_scores = cross_val_score(
        Pipeline([("sc", StandardScaler()), ("clf", base_clf)]),
        X, y, cv=cv, scoring="roc_auc_ovr_weighted"
    )
    print(f"CV ROC-AUC (OvR, weighted) : {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")
    model.fit(X, y)
    print(f"Classes in model           : {model.classes_}  (0=Loss, 1=Draw, 2=Win for {team_a})")

In [ ]:
# ── prediction for MATCH_DATE ─────────────────────────────────────────────────
a_f   = wf[team_a].iloc[-1]
b_f   = wf[team_b].iloc[-1]
sq_max = max(squad_sizes.values())

pred_vec = np.array([[a_f[c] - b_f[c] for c in FCOLS] +
                     [0,   # neutral venue
                      (squad_sizes[team_a] - squad_sizes[team_b]) / sq_max]])

probs      = model.predict_proba(pred_vec)[0]
classes    = model.classes_           # [0, 1, 2]
prob_map   = dict(zip(classes, probs))

a_win_p  = prob_map.get(2, 0.0)   # Team A wins
draw_p   = prob_map.get(1, 0.0)   # Draw
b_win_p  = prob_map.get(0, 0.0)   # Team B wins (= Team A loss)

print("=" * 55)
print(f"  {team_a} vs {team_b}")
print(f"  {MATCH_DATE}")
print("=" * 55)
print(f"  {team_a:<22} win  : {a_win_p:>6.1%}")
print(f"  {'Draw':<22}      : {draw_p:>6.1%}")
print(f"  {team_b:<22} win  : {b_win_p:>6.1%}")
print("=" * 55)

# Implied decimal odds
print("\nImplied decimal odds:")
for label, p in [(f"{team_a} win", a_win_p), ("Draw", draw_p), (f"{team_b} win", b_win_p)]:
    odds = round(1 / p, 2) if p > 0 else float("inf")
    print(f"  {label:<22} : {odds}")

In [ ]:
# ── visualizations ────────────────────────────────────────────────────────────
col_a  = team_color(team_a, 0)
col_b  = team_color(team_b, 1)
col_d  = "#AAAAAA"   # draw colour

fig = plt.figure(figsize=(15, 11))
fig.suptitle(f"{team_a} vs {team_b} — {MATCH_DATE}",
             fontsize=14, fontweight="bold")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.48, wspace=0.42)

# 1. 3-way result probability donut
ax1 = fig.add_subplot(gs[0, 0])
donut_vals   = [a_win_p, draw_p, b_win_p]
donut_labels = [f"{team_a}\n{a_win_p:.1%}",
                f"Draw\n{draw_p:.1%}",
                f"{team_b}\n{b_win_p:.1%}"]
wedges, _ = ax1.pie(
    donut_vals,
    colors=[col_a, col_d, col_b],
    startangle=90,
    wedgeprops={"width": 0.55, "edgecolor": "white"},
)
ax1.legend(wedges, donut_labels,
           loc="lower center", bbox_to_anchor=(0.5, -0.18), fontsize=8)
ax1.set_title("Result Probability (Win / Draw / Loss)")

# 2. Competition mix
ax2 = fig.add_subplot(gs[0, 1])
top_comps = list(COMP_WEIGHTS.keys())
a_cnt = dfs[team_a]["comp"].value_counts().reindex(top_comps, fill_value=0)
b_cnt = dfs[team_b]["comp"].value_counts().reindex(top_comps, fill_value=0)
x_c   = np.arange(len(top_comps))
ax2.bar(x_c - 0.2, a_cnt.values, width=0.4, label=team_a, color=col_a, alpha=0.85)
ax2.bar(x_c + 0.2, b_cnt.values, width=0.4, label=team_b, color=col_b, alpha=0.85)
ax2.set_xticks(x_c); ax2.set_xticklabels(top_comps, rotation=45, ha="right")
ax2.set_title("Match Count by Competition")
ax2.legend(); ax2.spines[["top", "right"]].set_visible(False)

# 3. Aggregate stats comparison
ax3 = fig.add_subplot(gs[0, 2])
cats3 = ["GF/g", "GA/g", "Win%", "Draw%", "Pts/g"]
keys3 = ["gf_per_game", "ga_per_game", "win_rate", "draw_rate", "pts_per_game"]
a3    = [stats[team_a][k] for k in keys3]
b3    = [stats[team_b][k] for k in keys3]
x3    = np.arange(len(cats3))
ax3.bar(x3 - 0.2, a3, width=0.4, label=team_a, color=col_a, alpha=0.85)
ax3.bar(x3 + 0.2, b3, width=0.4, label=team_b, color=col_b, alpha=0.85)
ax3.set_xticks(x3); ax3.set_xticklabels(cats3, rotation=20, ha="right")
ax3.set_title("Weighted Aggregate Stats")
ax3.legend(); ax3.spines[["top", "right"]].set_visible(False)

# 4. Rolling win rate over time
ax4 = fig.add_subplot(gs[1, :2])
ax4.plot(wf[team_a]["date"], wf[team_a]["win_rate"],
         label=f"{team_a} Win%",  color=col_a, lw=2)
ax4.plot(wf[team_a]["date"], wf[team_a]["draw_rate"],
         label=f"{team_a} Draw%", color=col_a, lw=1.5, linestyle=":")
ax4.plot(wf[team_b]["date"], wf[team_b]["win_rate"],
         label=f"{team_b} Win%",  color=col_b, lw=2)
ax4.plot(wf[team_b]["date"], wf[team_b]["draw_rate"],
         label=f"{team_b} Draw%", color=col_b, lw=1.5, linestyle=":")
ax4.set_title(f"Rolling Form — Win% and Draw% (window={WINDOW}, competition-weighted)")
ax4.set_ylabel("Rate")
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax4.legend(fontsize=8); ax4.grid(axis="y", alpha=0.3)
ax4.spines[["top", "right"]].set_visible(False)

# 5. GF / GA trend
ax5 = fig.add_subplot(gs[1, 2])
ax5.plot(wf[team_a]["date"], wf[team_a]["gf_rate"],
         color=col_a, lw=2,   label=f"{team_a} GF")
ax5.plot(wf[team_a]["date"], wf[team_a]["ga_rate"],
         color=col_a, lw=1.5, linestyle="--", label=f"{team_a} GA")
ax5.plot(wf[team_b]["date"], wf[team_b]["gf_rate"],
         color=col_b, lw=2,   label=f"{team_b} GF")
ax5.plot(wf[team_b]["date"], wf[team_b]["ga_rate"],
         color=col_b, lw=1.5, linestyle="--", label=f"{team_b} GA")
ax5.set_title("Weighted GF / GA per Game")
ax5.legend(fontsize=7); ax5.grid(axis="y", alpha=0.3)
ax5.spines[["top", "right"]].set_visible(False)

out_img = (
    f"{team_a.replace(' ', '_')}_vs_{team_b.replace(' ', '_')}_output.png"
)
plt.savefig(out_img, dpi=130, bbox_inches="tight")
plt.show()
print(f"saved → {out_img}")

In [ ]:
# ── sensitivity analysis: WC weight multiplier ────────────────────────────────
results = []
for wc_boost in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    local_w = COMP_WEIGHTS.copy()
    local_w["WC"] = wc_boost

    dfs_l = {n: parse_matches(raw[n], tid, comp_weights=local_w)
             for n, tid in TEAM_IDS.items()}
    wf_l  = {n: windowed_features(dfs_l[n]) for n in TEAM_IDS}
    tr_l  = build_diff_dataset(
        wf_l[team_a], wf_l[team_b],
        squad_sizes[team_a], squad_sizes[team_b]
    )
    Xl, yl = tr_l[FEATS].values, tr_l["label"].values

    cc  = Counter(yl)
    cvs = min(5, min(cc.values())) if len(cc) == 3 else 0
    if cvs < 2:
        continue

    m_l = CalibratedClassifierCV(
        Pipeline([("sc", StandardScaler()),
                  ("clf", LogisticRegression(C=1.5, max_iter=1000, random_state=99))]),
        cv=cvs, method="sigmoid",
    )
    m_l.fit(Xl, yl)

    af = wf_l[team_a].iloc[-1]
    bf = wf_l[team_b].iloc[-1]
    pv = np.array([[af[c] - bf[c] for c in FCOLS] + [0, pred_vec[0, -1]]])
    p  = dict(zip(m_l.classes_, m_l.predict_proba(pv)[0]))

    results.append({
        "WC_weight":         wc_boost,
        f"{team_a}_win":     p.get(2, 0.0),
        "draw":              p.get(1, 0.0),
        f"{team_b}_win":     p.get(0, 0.0),
    })

if results:
    sens_df = pd.DataFrame(results)
    print("\nSensitivity to WC match weight:")
    print(sens_df.round(3).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(sens_df["WC_weight"], sens_df[f"{team_a}_win"],
            "o-", color=col_a, lw=2, label=f"{team_a} win")
    ax.plot(sens_df["WC_weight"], sens_df["draw"],
            "o-", color=col_d, lw=2, label="Draw")
    ax.plot(sens_df["WC_weight"], sens_df[f"{team_b}_win"],
            "o-", color=col_b, lw=2, label=f"{team_b} win")
    ax.set_xlabel("WC competition weight multiplier")
    ax.set_ylabel("Probability")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.set_title(f"Sensitivity: WC weight vs Result Probability ({team_a} vs {team_b})")
    ax.legend(); ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    sens_img = (
        f"{team_a.replace(' ', '_')}_vs_{team_b.replace(' ', '_')}_sensitivity.png"
    )
    plt.savefig(sens_img, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"saved → {sens_img}")
else:
    print("Sensitivity analysis skipped — insufficient class diversity across WC weight sweep.")